# SBCSAE Gold Annotation Pipeline

Parses the **Santa Barbara Corpus of Spoken American English** transcripts into
overlapping IU-window samples with gold-standard prosodic boundary and intonation labels,
and writes structured batch label files to Google Drive.

Labels are derived **entirely from the Du Bois transcript markup** — no audio, no models.
Boundary labels come from IU line boundaries; intonation labels come from terminal
contour markers (`.` falling, `,` level, `?` rising).

Disclaimer: Code was largely generated with the help of Claude Sonnet 3.5 (Anthropic, 2026 February - May). Prompts, code tweaks and verification by me.

---

## How to use this notebook

Run cells **top to bottom**. The only cell you should need to edit is **Cell 1 (Configuration)**.

| Section | Cell | What it does |
|---|---|---|
| **1 · Configuration** | 1 | Set paths, window params, noise settings |
| **2 · Environment** | 2 | Mount Drive, create dirs, download + unzip transcripts |
| **3 · Helper Functions** | 3–6 | Noise stripping, parsing, windowing, batch I/O |
| **4 · Run Pipeline** | 7 | Main loop — parse all 60 files, write batch JSONs |
| **5 · Analysis** | 8 | Summary statistics + write meta.json |

---

## Output folder structure

```
labels/
└── sbcsae/
    ├── meta.json
    ├── batch_0000.json
    ├── batch_0001.json
    └── ...
```

Each batch file contains up to 1000 samples keyed by `SBC{NNN}_w{NNNN}`.

---

## Label schema (per sample)

```json
{
  "SBC001_w0000": {
    "b": { "tokens": [...], "consensus": [0,0,1,...] },
    "i": { "tokens": [...], "labels":    [0,0,2,...] },
    "x": null
  }
}
```

- `b.consensus` — `1` at the last word of every IU (gold boundary), `0` elsewhere.
  Named `consensus` for drop-in compatibility with the DistilBERT loader.
- `i.labels`    — intonation at each boundary position; `0` elsewhere.
  Mapping: `{none:0, rising:1, falling:2, level:3}`.
- `x`           — `null` (ToBI break indices not available from Du Bois transcripts).

---

## Key design decisions

- **Windows** are 30 consecutive IUs with a stride of 15 (50 % overlap).
  Every IU therefore appears in exactly two windows (except at conversation edges).
- Windows cross speaker turns naturally — speaker identity is never shown to the model.
- **Noise stripping** removes all non-lexical Du Bois markers by default.
  `STRIP_FILLERS` controls whether filled pauses (um/uh) are also stripped.
- Test split (SBC001–005) is **not** excluded here; the DistilBERT notebook handles splitting.
  Sample IDs encode the source file number for easy filtering.


---
## Section 1 · Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  This is the only cell you need to edit between runs.        ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Paths ────────────────────────────────────────────────────────
DRIVE_ROOT          = "/content/drive/MyDrive/Capstone/project"
SBC_LABELS_ROOT     = f"{DRIVE_ROOT}/labels/sbcsae"
CLEAN_360_ROOT      = f"{DRIVE_ROOT}/labels/clean-360"

# Transcripts are downloaded here at session start (Colab local SSD, not Drive)
# and discarded when the runtime ends.  Drive is only written to for label output.
LOCAL_TRANSCRIPTS   = "/content/sbcsae_transcripts"

# ── Windowing ────────────────────────────────────────────────────
WINDOW_SIZE         = 30    # IUs per window
WINDOW_STRIDE       = 15    # IUs between window starts (50 % overlap)
MIN_TOKENS_WINDOW   = 10    # discard windows with fewer tokens than this

# ── Batching ─────────────────────────────────────────────────────
FILE_BATCH_SIZE     = 100  # samples per batch JSON on Drive

# ── Noise stripping ──────────────────────────────────────────────
# Set STRIP_FILLERS = False to keep 'um', 'uh', 'mm', 'mhm', 'uh-huh'.
# Set STRIP_PUNCTUATION = False to retain punctuation on word tokens.
STRIP_FILLERS       = False
STRIP_PUNCTUATION   = True

# ── Processing control ───────────────────────────────────────────
FORCE_REPROCESS      = True
MARK_SPEAKER_CHANGES = True

print("✓ Configuration loaded.")
print(f"  Output:          {SBC_LABELS_ROOT}")
print(f"  Window:          {WINDOW_SIZE} IUs  |  stride: {WINDOW_STRIDE}  |  min tokens: {MIN_TOKENS_WINDOW}")
print(f"  Batch size:      {FILE_BATCH_SIZE} samples/file")
print(f"  Strip fillers:   {STRIP_FILLERS}")
print(f"  Strip punct:     {STRIP_PUNCTUATION}")
print(f"  Force reprocess: {FORCE_REPROCESS}")
print(f"  Mark speaker changes: {MARK_SPEAKER_CHANGES}")

---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + download transcripts                 ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os, glob

drive.mount("/content/drive", force_remount=True)

# Create Drive output directory
os.makedirs(SBC_LABELS_ROOT, exist_ok=True)
print("✓ Drive mounted. Output directory ready.")

# ── Download transcripts to local SSD ────────────────────────────
# Transcripts are small (~few MB) and parse in seconds from local disk.
# We never write them to Drive — they are re-downloaded each session.
TRN_ZIP_URL = (
    "https://www.linguistics.ucsb.edu/sites/default/files/"
    "sitefiles/research/SBC/SBCorpus.zip"
)
ZIP_LOCAL = "/content/SBCorpus.zip"

if not os.path.isdir(LOCAL_TRANSCRIPTS) or \
        len(glob.glob(os.path.join(LOCAL_TRANSCRIPTS, '**', '*.trn'), recursive=True)) == 0:
    print("Downloading SBCorpus.zip ...")
    !wget -q --show-progress -O {ZIP_LOCAL} {TRN_ZIP_URL}
    os.makedirs(LOCAL_TRANSCRIPTS, exist_ok=True)
    !unzip -q -o {ZIP_LOCAL} -d {LOCAL_TRANSCRIPTS}
    print("✓ Unzipped.")
else:
    print("✓ Transcripts already present — skipping download.")

# ── Locate .trn files ─────────────────────────────────────────────
trn_files = sorted(
    glob.glob(os.path.join(LOCAL_TRANSCRIPTS, '**', 'SBC*.trn'), recursive=True)
)
# Fallback: some distributions use uppercase or no SBC prefix
if not trn_files:
    trn_files = sorted(
        glob.glob(os.path.join(LOCAL_TRANSCRIPTS, '**', '*.trn'), recursive=True)
    )

if not trn_files:
    print("\n⚠  No .trn files found after extraction.")
    print("   The UCSB download may have been blocked. Options:")
    print("   1. Download SBCorpus.zip manually and upload to Drive, then copy to /content/")
    print("   2. Use the OpenSLR mirror: https://openslr.trmal.net/resources/155/SBCSAE.tar.gz")
    print("      (6.1 GB — audio + transcripts; extract only the .trn files)")
else:
    print(f"\n✓ Found {len(trn_files)} .trn files.")
    for f in trn_files[:3]:
        print(f"  {os.path.basename(f)}")
    if len(trn_files) > 3:
        print(f"  ... and {len(trn_files) - 3} more")

---
## Section 3 · Helper Functions

Define all processing logic. No edits needed.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Noise stripping + contour extraction               ║
# ╚══════════════════════════════════════════════════════════════╝
import re

# ── Label maps ───────────────────────────────────────────────────
INTONATION_MAP = {'none': 0, 'rising': 1, 'falling': 2, 'level': 3, 'unclear': 4}

# Du Bois terminal contour markers → intonation label
CONTOUR_MAP = {
    '.': 2,   # final / falling
    ',': 3,   # continuing / level
    '?': 1,   # appeal / rising
    '!': 1,   # emphatic rising (treat as rising)
}

# ── Noise patterns (always stripped) ─────────────────────────────
# Each entry is (compiled_regex, replacement_string).
# Order matters: anonymization and vocal-quality tags first,
# then inline markers, then residual whitespace.
_NOISE = [
    (re.compile(r'\[\[.*?\]\]'),                    ''),   # anonymization [[NAME]]
    (re.compile(r'\(\(.*?\)\)'),                    ''),   # stage directions ((ENV_SOUND))  ← NEW
    (re.compile(r'</?[A-Z]+[^>]*>'),                ''),   # opening vocal quality tags <SM>, <VOX>
    (re.compile(r'\b[A-Z]{1,5}>'),                  ''),   # closing symmetric tags SM>, HI>, VOX>  ← NEW
    (re.compile(r'\(H[Xx]?\)'),                     ''),   # breath
    (re.compile(r'\(TSK\)', re.I),                  ''),   # click
    (re.compile(r'\(SNIFF\)', re.I),                ''),   # sniff
    (re.compile(r'\((THROAT|SWALLOW|COUGH|MICROPHONE|SNORT|PARP|PPAR)\)', re.I), ''),
    (re.compile(r'@+'),                             ''),   # laughter
    (re.compile(r'\(\.{2,}\)'),                     ''),   # parenthesised pause (..)
    (re.compile(r'\.{2,}'),                         ''),   # bare pause dots
    (re.compile(r'\(\d+\.\d+\)'),                   ''),   # timed pauses (0.3)
    (re.compile(r'\[+\s*\d+'),                      ''),   # overlap start [1 [2
    (re.compile(r'\d+\]+'),                         ''),   # overlap close 2] 3]  ← NEW (replaces \]+\s*\d+)
    (re.compile(r'\]+\s*\d+'),                      ''),   # overlap close ]1 ]2 (keep for safety)
    (re.compile(r'\S*--+'),                         ''),   # truncated words
    (re.compile(r'\bX+\b'),                         ''),   # unintelligible
    (re.compile(r'='),                              ''),   # lengthening
    (re.compile(r'\bYWN\b'),                        ''),   # yawn tag residual
    (re.compile(r'~'),                              ''),   # unclear transcription marker
]

# Filled pauses — stripped only when STRIP_FILLERS = True
_FILLER_RE = re.compile(
    r'\b(um+|uh+|mm+|mhm+|uh-huh|hm+)\b', re.IGNORECASE
)

# Punctuation stripped from individual tokens when STRIP_PUNCTUATION = True
# Apostrophes and internal hyphens are preserved (contractions, compound words)
_PUNCT_STRIP_RE  = re.compile(r"[^\w'\-]")
_EDGE_HYPHEN_RE  = re.compile(r"^['-]+|['-]+$")


def extract_contour(raw_text):
    """
    Extract the terminal Du Bois contour marker from the end of an IU line.
    Returns (intonation_int, text_with_marker_removed).

    The contour marker is always the last non-whitespace character when present.
    Single dots are contour markers; double/triple dots are pause markers (handled
    separately in noise stripping).
    """
    text = raw_text.rstrip()
    if not text:
        return 0, text
    last_char = text[-1]
    if last_char in CONTOUR_MAP:
        # Guard: don't consume the last dot of '...' (pause marker)
        if last_char == '.' and len(text) >= 2 and text[-2] == '.':
            return 0, text   # trailing dots — no contour marker
        return CONTOUR_MAP[last_char], text[:-1].rstrip()
    return 0, text


def strip_noise(text):
    """Apply all noise patterns and return cleaned text."""
    for pattern, repl in _NOISE:
        text = pattern.sub(repl, text)
    if STRIP_FILLERS:
        text = _FILLER_RE.sub('', text)
    return text


def tokenize_iu(text):
    """
    Split cleaned IU text into word tokens.
    Optionally strips punctuation from token edges while preserving
    apostrophes (contractions) and internal hyphens (compound words).
    Empty tokens and single-character punctuation artifacts are discarded.
    """
    tokens = []
    for raw_tok in text.split():
        if STRIP_PUNCTUATION:
            tok = _PUNCT_STRIP_RE.sub('', raw_tok)
            tok = _EDGE_HYPHEN_RE.sub('', tok)
        else:
            tok = raw_tok
        tok = re.sub(r'\d+$', '', tok)   # strip trailing overlap tier numbers  ← NEW
        tok = tok.lstrip('_')            # strip latching marker  ← NEW
        if tok and not tok.strip("'-") == '':
            tokens.append(tok)
    return tokens


print("✓ Cell 3: Noise stripping + contour extraction helpers defined.")

# ── Quick smoke test ──────────────────────────────────────────────
_test_cases = [
    ("okay so I was thinking,",  3, ['okay', 'so', 'I', 'was', 'thinking']),
    ("what do you mean ?",       1, ['what', 'do', 'you', 'mean']),
    ("and that was it.",         2, ['and', 'that', 'was', 'it']),
    ("(H) yeah,",                3, ['yeah']),
    ("going-- to the,",          3, ['to', 'the']),
    ("ma=d about it.",           2, ['mad', 'about', 'it']),
]
all_pass = True
for raw, exp_inton, exp_toks in _test_cases:
    inton, stripped = extract_contour(raw)
    toks = tokenize_iu(strip_noise(stripped))
    ok = (inton == exp_inton) and (toks == exp_toks)
    if not ok:
        print(f"  ✗ FAIL  [{raw!r}]  inton={inton} (exp {exp_inton})  toks={toks} (exp {exp_toks})")
        all_pass = False
if all_pass:
    print("  All smoke tests passed.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Transcript parser (fixed for actual .trn format)   ║
# ╚══════════════════════════════════════════════════════════════╝

# format: START END\tSPEAKER:\tTEXT
# Continuation lines use spaces instead of a speaker name.
_LINE_RE = re.compile(
    r'^([\d.]+)\s+([\d.]+)\t(.*?)\t(.*)', re.UNICODE
)

def parse_trn_file(path):
    ius = []
    last_speaker = 'UNK'

    with open(path, encoding='utf-8', errors='replace') as fh:
        for raw_line in fh:
            line = raw_line.rstrip('\n')
            if not line.strip() or line.strip().startswith('#'):
                continue

            m = _LINE_RE.match(line)
            if not m:
                continue

            try:
                start = float(m.group(1))
                end   = float(m.group(2))
            except ValueError:
                continue

            raw_speaker = m.group(3).strip()
            if raw_speaker:
                last_speaker = raw_speaker.rstrip(':').strip()
            speaker = last_speaker

            raw_text = m.group(4).strip()

            # 1. Extract terminal contour BEFORE any stripping
            inton_label, text_no_contour = extract_contour(raw_text)

            # 2. Strip noise markers
            clean_text = strip_noise(text_no_contour)

            # 3. Tokenize
            tokens = tokenize_iu(clean_text)

            if not tokens:
                continue

            ius.append({
                'speaker':    speaker,
                'start':      start,
                'end':        end,
                'tokens':     tokens,
                'intonation': inton_label,
            })

    return ius


print("✓ Cell 4: Transcript parser defined.")

if trn_files:
    _sample_path = trn_files[0]
    _sample_ius  = parse_trn_file(_sample_path)
    _fname       = os.path.basename(_sample_path)
    print(f"\nSpot-check: {_fname}  →  {len(_sample_ius)} IUs after stripping")
    print(f"  First 3 IUs:")
    for iu in _sample_ius[:3]:
        inton_name = {v: k for k, v in INTONATION_MAP.items()}.get(iu['intonation'], '?')
        print(f"    [{iu['speaker']:8s}] {iu['tokens']}  →  {inton_name}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Window generation (speaker-change token support)  ║
# ╚══════════════════════════════════════════════════════════════╝

SPK_CHANGE_TOKEN = "/"   # injected at every speaker turn boundary

def make_windows(ius, file_id,
                 window_size=WINDOW_SIZE,
                 stride=WINDOW_STRIDE,
                 min_tokens=MIN_TOKENS_WINDOW,
                 mark_speaker_changes=MARK_SPEAKER_CHANGES):
    """
    Slide a window of `window_size` IUs over the full IU list.

    When `mark_speaker_changes=True`, a "/" token is inserted between
    consecutive IUs that belong to different speakers.  This token:
      - receives b_label=1  (guaranteed prosodic boundary)
      - receives i_label=0  (no intonation — maps to -100/masked in
                             ProsodyDataset because "none"=0 is masked)

    The IU's actual last word still carries b_label=1 as normal; "/"
    is an *additional* boundary token in the seam.  At eval time the
    DistilBERT notebook excludes these positions from F1 so they do
    not inflate metrics — but they remain supervised during training
    so the model explicitly learns the "/" → boundary hard rule.
    """
    windows = []
    n       = len(ius)
    w_idx   = 0

    for start in range(0, n, stride):
        end        = min(start + window_size, n)
        window_ius = ius[start:end]

        tokens, b_labels, i_labels = [], [], []
        prev_speaker = None

        for iu in window_ius:
            # ── Speaker-change marker ──────────────────────────────────
            if (mark_speaker_changes
                    and prev_speaker is not None
                    and iu['speaker'] != prev_speaker):
                tokens.append(SPK_CHANGE_TOKEN)
                b_labels.append(1)   # guaranteed boundary
                i_labels.append(0)   # 0 = "none" → masked in _process

            # ── Normal IU tokens ───────────────────────────────────────
            n_toks = len(iu['tokens'])
            for j, tok in enumerate(iu['tokens']):
                tokens.append(tok)
                is_boundary = (j == n_toks - 1)
                b_labels.append(1 if is_boundary else 0)
                i_labels.append(iu['intonation'] if is_boundary else 0)

            prev_speaker = iu['speaker']

        if len(tokens) < min_tokens:
            continue

        windows.append({
            'sample_id': f"{file_id}_w{w_idx:04d}",
            'tokens':    tokens,
            'b_labels':  b_labels,
            'i_labels':  i_labels,
        })
        w_idx += 1

    return windows


print("✓ Cell 5: Window generation defined (speaker-change markers active).")

# ── Spot-check ────────────────────────────────────────────────────
if trn_files:
    _test_wins = make_windows(_sample_ius, 'SBC001')
    print(f"  {os.path.basename(trn_files[0])} → {len(_sample_ius)} IUs → {len(_test_wins)} windows")
    _w0 = _test_wins[0]
    print(f"  Window 0 ({_w0['sample_id']}):")
    print(f"    {len(_w0['tokens'])} tokens  |  "
          f"{sum(_w0['b_labels'])} boundaries  |  "
          f"first 8 tokens: {_w0['tokens'][:8]}")
    # Verify overlap: IU 15 of window 0 should equal IU 0 of window 1
    if len(_test_wins) >= 2:
        _overlap_iu_tokens = [t for t, b in zip(_test_wins[0]['tokens'],
                                                 _test_wins[0]['b_labels'])
                               if b == 1][WINDOW_STRIDE - 1 : WINDOW_STRIDE]
        print(f"    Overlap check (IU-{WINDOW_STRIDE} boundary in w0 == IU-0 boundary in w1): ", end='')
        # boundary count at stride point should match
        _b0_count = sum(_test_wins[0]['b_labels'])
        _b1_count = sum(_test_wins[1]['b_labels'])
        print(f"w0 has {_b0_count} boundaries, w1 has {_b1_count} — overlap shared IUs: {WINDOW_STRIDE}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Batch I/O helpers                                  ║
# ╚══════════════════════════════════════════════════════════════╝
import json

_batch_buffer = {}    # {sample_id: {"b": ..., "i": ..., "x": None}}
_batch_index  = 0     # index of the next batch file to write
_done_ids     = set() # all sample IDs already written to disk


def _init_batch_state():
    """Scan existing batch files on startup to support session resuming."""
    global _batch_index, _done_ids
    os.makedirs(SBC_LABELS_ROOT, exist_ok=True)
    existing = sorted(
        f for f in os.listdir(SBC_LABELS_ROOT)
        if f.startswith('batch_') and f.endswith('.json')
    )
    for fname in existing:
        with open(os.path.join(SBC_LABELS_ROOT, fname)) as fh:
            _done_ids.update(json.load(fh).keys())
    _batch_index = len(existing)
    print(f"  Batch state: {len(existing)} existing file(s), "
          f"{len(_done_ids)} samples already done.")


def _flush_buffer():
    global _batch_buffer, _batch_index
    if not _batch_buffer:
        return
    fname = f"batch_{_batch_index:04d}.json"
    path  = os.path.join(SBC_LABELS_ROOT, fname)

    # Dump with indent, then collapse arrays onto single lines
    raw = json.dumps(_batch_buffer, indent=2)
    raw = re.sub(r'\[\n\s+(.*?)\n\s+\]',
                 lambda m: '[' + re.sub(r'\s*\n\s*', ' ', m.group(1)) + ']',
                 raw, flags=re.DOTALL)

    with open(path, 'w') as fh:
        fh.write(raw)
    print(f"  → Flushed {len(_batch_buffer)} samples → {fname}")
    _batch_buffer = {}
    _batch_index += 1


def add_to_batch(sample_id, tokens, b_labels, i_labels):
    """
    Stage one sample in the buffer.  Flushes to disk automatically when
    the buffer reaches FILE_BATCH_SIZE.

    Schema mirrors the LibriTTS batched labels for DistilBERT loader compatibility:
      b.consensus  — boundary labels (gold, named 'consensus' for loader compatibility)
      i.labels     — intonation labels
      x            — null (ToBI break indices not available from transcripts)
    """
    _batch_buffer[sample_id] = {
        'b': {
            'tokens':    tokens,
            'consensus': b_labels,
        },
        'i': {
            'tokens': tokens,
            'labels': i_labels,
        },
        'x': None,
    }
    _done_ids.add(sample_id)
    if len(_batch_buffer) >= FILE_BATCH_SIZE:
        _flush_buffer()


def flush_remaining():
    """Flush any samples left in the buffer at end of processing."""
    if _batch_buffer:
        _flush_buffer()


print("✓ Cell 6: Batch I/O helpers defined.")

---
## Section 4 · Run Pipeline

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Main processing loop                               ║
# ║                                                              ║
# ║  For each .trn file:                                         ║
# ║    1. Parse IUs from transcript                              ║
# ║    2. Slide window → samples                                 ║
# ║    3. Stage samples in batch buffer (flush at 1000)          ║
# ║                                                              ║
# ║  Already-processed sample IDs are skipped unless             ║
# ║  FORCE_REPROCESS = True.                                     ║
# ╚══════════════════════════════════════════════════════════════╝
import re as _re

_init_batch_state()

if FORCE_REPROCESS:
    _done_ids.clear()
    print("  FORCE_REPROCESS=True — all samples will be rewritten.")

total_files      = 0
total_ius        = 0
total_windows    = 0
total_new        = 0
total_tokens     = 0
total_boundaries = 0

print(f"\nProcessing {len(trn_files)} files...\n")

for trn_path in trn_files:
    basename = os.path.basename(trn_path)

    # Derive canonical file ID: SBC001 … SBC060
    _num = _re.search(r'(\d+)', basename)
    if not _num:
        print(f"  {basename} — ⚠ cannot parse file number, skipping")
        continue
    file_id = f"SBC{int(_num.group(1)):03d}"

    # ── Parse ──────────────────────────────────────────────────
    ius = parse_trn_file(trn_path)
    if not ius:
        print(f"  {file_id} — ⚠ no IUs parsed, skipping")
        continue

    # ── Window ─────────────────────────────────────────────────
    windows = make_windows(ius, file_id)

    # ── Stage ──────────────────────────────────────────────────
    n_new = 0
    for w in windows:
        if w['sample_id'] in _done_ids and not FORCE_REPROCESS:
            continue
        add_to_batch(w['sample_id'], w['tokens'], w['b_labels'], w['i_labels'])
        n_new          += 1
        total_tokens   += len(w['tokens'])
        total_boundaries += sum(w['b_labels'])

    total_files   += 1
    total_ius     += len(ius)
    total_windows += len(windows)
    total_new     += n_new

    print(f"  {file_id} — {len(ius):4d} IUs → {len(windows):4d} windows "
          f"(+{n_new} new)")

flush_remaining()

print(f"\n{'─'*55}")
print(f"✓ Done.")
print(f"  Files processed:  {total_files}")
print(f"  Total IUs:        {total_ius:,}")
print(f"  Total windows:    {total_windows:,}")
print(f"  New this run:     {total_new:,}")
print(f"  Total tokens:     {total_tokens:,}")
print(f"  Total boundaries: {total_boundaries:,}")
if total_tokens > 0:
    print(f"  Boundary rate:    {total_boundaries/total_tokens:.3f}")

---
## Section 5 · Analysis & Meta

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Summary statistics + write meta.json               ║
# ╚══════════════════════════════════════════════════════════════╝
from collections import Counter
from datetime import datetime

# ── Load all batch files ──────────────────────────────────────────
all_samples = {}
batch_files = sorted(
    f for f in os.listdir(SBC_LABELS_ROOT)
    if f.startswith('batch_') and f.endswith('.json')
)
for bfname in batch_files:
    with open(os.path.join(SBC_LABELS_ROOT, bfname)) as fh:
        all_samples.update(json.load(fh))

# ── Aggregate stats ───────────────────────────────────────────────
n_samples    = len(all_samples)
n_tokens     = sum(len(v['b']['tokens'])    for v in all_samples.values())
n_boundaries = sum(sum(v['b']['consensus']) for v in all_samples.values())

inton_counts  = Counter()
source_counts = Counter()
for sid, v in all_samples.items():
    inton_counts.update(v['i']['labels'])
    source_counts[sid[:6]] += 1   # SBC001 … SBC060

_INTON_NAMES = {v: k for k, v in INTONATION_MAP.items()}

print(f"{'─'*55}")
print(f"  Batch files:      {len(batch_files)}")
print(f"  Total samples:    {n_samples:,}")
print(f"  Total tokens:     {n_tokens:,}")
print(f"  Total boundaries: {n_boundaries:,}")
print(f"  Boundary rate:    {n_boundaries/max(n_tokens,1):.3f}  "
      f"(expected ~0.17 for 30-IU windows at ~5 words/IU)")
print()
print("  Intonation distribution at boundaries:")
for label in sorted(inton_counts):
    if label == 0:
        continue   # non-boundary positions — skip
    count = inton_counts[label]
    pct   = 100 * count / max(n_boundaries, 1)
    print(f"    {_INTON_NAMES.get(label,'?'):10s} ({label}): {count:6,}  ({pct:.1f}%)")
missing = n_boundaries - sum(v for k, v in inton_counts.items() if k != 0)
if missing > 0:
    print(f"    {'none':10s} (0): {missing:6,}  "
          f"(IUs with no terminal contour marker)")

print()
print("  Samples per file (first 5 = held-out test split):")
for file_id in sorted(source_counts)[:10]:
    marker = " ← TEST" if int(file_id[3:]) <= 5 else ""
    print(f"    {file_id}: {source_counts[file_id]:4d} windows{marker}")
if len(source_counts) > 10:
    print(f"    ... and {len(source_counts) - 10} more files")

# ── Write meta.json ───────────────────────────────────────────────
meta = {
    'source':            'SBCSAE',
    'label_type':        'gold_du_bois',
    'created':           datetime.now().isoformat(),
    'window_size':       WINDOW_SIZE,
    'window_stride':     WINDOW_STRIDE,
    'min_tokens_window': MIN_TOKENS_WINDOW,
    'file_batch_size':   FILE_BATCH_SIZE,
    'strip_fillers':     STRIP_FILLERS,
    'strip_punctuation': STRIP_PUNCTUATION,
    'n_batch_files':     len(batch_files),
    'n_samples':         n_samples,
    'n_tokens':          n_tokens,
    'n_boundaries':      n_boundaries,
    'boundary_rate':     round(n_boundaries / max(n_tokens, 1), 4),
    'intonation_counts': {_INTON_NAMES.get(k, k): v
                          for k, v in inton_counts.items()},
    'test_split_files':  ['SBC001','SBC002','SBC003','SBC004','SBC005'],
}
meta_path = os.path.join(SBC_LABELS_ROOT, 'meta.json')
with open(meta_path, 'w') as fh:
    json.dump(meta, fh, indent=2)

print(f"\n✓ meta.json written to {meta_path}")

In [ ]:
import re, json, os

for fname in sorted(os.listdir(SBC_LABELS_ROOT)):
    if not (fname.startswith('batch_') and fname.endswith('.json')):
        continue
    path = os.path.join(SBC_LABELS_ROOT, fname)
    with open(path) as f:
        data = json.load(f)
    raw = json.dumps(data, indent=2)
    raw = re.sub(r'\[\n\s+(.*?)\n\s+\]',
                 lambda m: '[' + re.sub(r'\s*\n\s*', ' ', m.group(1)) + ']',
                 raw, flags=re.DOTALL)
    with open(path, 'w') as f:
        f.write(raw)
    print(f"Reformatted {fname}")

In [ ]:
'''
with open('/content/sbcsae_transcripts/TRN/SBC017.trn') as f:
    print(f.read())
'''

In [ ]:
trn_files